# N066 · 切分回溯与字符串约束

**目标：** 把切分边界当作选择并维护合法前缀。

**先修：** N023, N064。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 切分树；回文判断；前导零；剩余长度剪枝；恢复。

**配套讲解来源：** [同名逐章意见](../../comment/066_partitioning_and_search_constraints.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章处理"把一根字符串切成几段"的回溯题，切的位置就是我们的"选择"。两道代表题：

**题一：回文切分。** 给你一个字符串 s，你要把它切成若干段，使得**每一段单独读起来都是回文**（正着读和倒着读一样），列出所有切法。比如 s=`'aab'`，答案是两种：`['a','a','b']`（切成三个单字符，单个字符当然是回文）和 `['aa','b']`（'aa' 是回文）。为什么没有 `['a','ab']`？因为 'ab' 反过来是 'ba'，不是回文。

**题二：复原 IP 地址。** 给你一串纯数字，比如 `'25525511135'`，你要在里面插三个点把它变成合法的 IPv4 地址。合法的标准有四条：必须恰好四段；每段长度 1 到 3；每段的数值在 0..255 之间；每段不能有"多余的前导零"（'0' 合法，'01'、'00' 非法）。`'25525511135'` 的答案是 `'255.255.11.135'` 和 `'255.255.111.35'` 两个。再比如 `'0000'` 只能切出 `'0.0.0.0'`，而不是 `'00.0.0.0'`（前导零违规）。

两道题的共同骨架：状态是"已经处理到哪个下标"，每一轮的选择是"下一段在哪里结束"。不同点在于每道题挂在选择上的约束不同：回文切分要求切出来的段是回文，IP 要求段满足数字规则。

## 2. 基础知识：先读懂这些词

- **切分树**：把"每一段的结束位置"当作一次选择画成的树。树的第 d 层对应第 d 段的切法，从根到叶的一条路径就是一种完整切分。
- **状态（start）**：当前已经切到哪里。`visit(start)` 表示"s[start:] 这段后缀还没切"，我们只需要关心这个下标，前面的历史不影响后面能怎么切。
- **回文（palindrome）**：正读反读相同的字符串，比如 'a'、'aa'、'aba'。判断方法是最朴素的双指针：左右两端字符相同，且去掉两端后中间那截也是回文。
- **回文预计算表**：一个二维表 `palindrome[left][right]`，存"s 的第 left 到 right 个字符组成的段是不是回文"。提前算好，回溯时查表是 O(1)，避免同一个段被反复判断。它的递推依赖是"大段回文 ⇔ 两端相等且内部小段回文"，所以填表要按 left 从右往左的顺序。
- **前导零**：段的开头是 '0' 但长度大于 1，比如 '01'。IP 规范里它非法（会被当成 1，产生歧义），所以切分时要挡掉。
- **剩余长度剪枝**：IP 有四段、每段最多 3 个字符，所以任何时刻"还没切的字符数"必须落在"剩余段数 × 1"到"剩余段数 × 3"之间，超出就直接放弃这个分支。这一刀能把 13 个字符以上的输入在根部直接判死。
- **恢复（undo）**：老规矩，`path.append(...)` 探索完要 `path.pop()`，让兄弟分支拿到干净的 path。

## 3. 思路推导：从小例子开始

**Step 1：把"切分"翻译成"选下一段的终点"。** 在 `visit(start)` 里，我们枚举 `end` 从 start 到 n-1，候选段就是 `s[start:end+1]`；这个段合法，就把它接进 path，递归处理 `visit(end+1)`。

**Step 2：回文切分先填表。** 判断"一段是不是回文"会在递归里被反复问到（比如 'aa' 在不同分支里都会被问到），所以我们先用双重循环把所有区间的答案算好存进表。

**Step 3：IP 的约束拆成三道闸门。** 每一段候选要先过三关：长度 1..3、无多余前导零、数值 ≤255；全过才递归。另有一个全局的"剩余长度必须够填满剩余段、也不能多出来"的剪枝。

**Step 4：收集答案的位置不同。** 回文切分在 start 走到字符串末尾时收集段列表；IP 在凑满四段且字符恰好用完时把 path 用 '.' 拼成字符串收集。

**手算演示 1：给 'aab' 填回文表（n=3，下标 0,1,2）。** 填表顺序是 left 从 2 到 0：

| 区间 | 段 | 计算 | 结果 |
|---|---|---|---|
| [2,2] | 'b' | 两端同（都是 b），长度差 0<2 | True |
| [1,1] | 'a' | 同上 | True |
| [1,2] | 'ab' | s[1]='a' ≠ s[2]='b' | False |
| [0,0] | 'a' | 单字符 | True |
| [0,1] | 'aa' | 两端同为 a，长度差 1<2，不用看内部 | True |
| [0,2] | 'aab' | s[0]='a' ≠ s[2]='b' | False |

**手算演示 2：在表上走 `partition_palindromes('aab')`（对应 “运行示例”部分 第一张表）。**

1. `visit(0)`：试 end=0，段 'a' 是 True → path=['a']，进入 `visit(1)`。
2. `visit(1)`：试 end=1，段 'a' True → path=['a','a'] → `visit(2)`：end=2，'b' True → path=['a','a','b'] → `visit(3)`：start==n，收集 `['a','a','b']`。回溯；`visit(1)` 再试 end=2，段 'ab' 是 False，跳过。
3. 回到 `visit(0)`：试 end=1，段 'aa' True → path=['aa'] → `visit(2)`：'b' True → 收集 `['aa','b']`；再试 end=2，段 'aab' False，跳过。
4. 最终 `[['a','a','b'],['aa','b']]`，与 “运行示例”部分 表格和 “自动测试”部分 断言完全一致（表里还附了每段长度 [1,1,2]/[2,1] 和"全为回文"的复核列）。

**手算演示 3：`restore_ip_addresses('0000')`（“运行示例”部分 第二张表）。** 根部 needed=4、remaining=4，满足 4≤4≤12。第一段试长度 1：'0' 合法（单字符零可以）→ 长度 2：'00' 前导零，break（注意是 break，更长的 '000' 也必然带前导零）。于是每段都只能取 '0'，唯一答案 '0.0.0.0'。

**手算演示 4：跟 `'25525511135'` 走一条完整分支，看剪枝怎么干活。** 字符串长 11，下标 0..10。

1. 第一段试长度 1（'2'）、长度 2（'25'）都能往下走但最终凑不满四段；试长度 3：'255' 合法（无前导零、值不超 255），进入第二段。
2. 第二段从下标 3 试起：'2'、'25'、'255' 中只有 '255' 这条活下来，进入第三段（下标 6，还剩 '11135' 五个字符）。
3. 第三段试 '1'（下标 6-6）：此时 path 已有三段，第四段 entry 检查 needed=1、remaining=len(s)-7=4，条件 1≤4≤3 不成立——**长度剪枝直接判死**，第四段连循环都没进（'1135' 有 4 个字符，塞不进最多 3 个字符的一段）。
4. 第三段改试 '11'（下标 6-7）：第四段 remaining=3，合法；试 '135'（下标 8-10）成功 → 收集 `255.255.11.135`。
5. 第三段试 '111'（下标 6-8）：第四段剩 '35'，'3'、'35' 都合法，其中 '35' → 收集 `255.255.111.35`。两个答案到齐，与 “自动测试”部分 断言一致。

你看，第 3 步就是"剩余长度剪枝"的现场表演：它没有任何花哨计算，只靠"每段 1..3 个字符"这条硬规则就把死分支掐灭在萌芽里。

## 4. 核心代码：partition_palindromes

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def partition_palindromes(s):
    n = len(s)
    palindrome = [[False] * n for _ in range(n)]
    for left in range(n - 1, -1, -1):
        for right in range(left, n):
            palindrome[left][right] = s[left] == s[right] and (right - left < 2 or palindrome[left + 1][right - 1])
    out = []
    path = []

    def visit(start):
        if start == n:
            out.append(path[:])
            return
        for end in range(start, n):
            if palindrome[start][end]:
                path.append(s[start:end + 1])
                visit(end + 1)
                path.pop()
    visit(0)
    return out
```

### 逐段读懂代码

### 4.1 `partition_palindromes(s)`

```python
def partition_palindromes(s):
    n = len(s)
    palindrome = [[False] * n for _ in range(n)]
    for left in range(n - 1, -1, -1):
        for right in range(left, n):
            palindrome[left][right] = s[left] == s[right] and (right - left < 2 or palindrome[left + 1][right - 1])
    out = []
    path = []

    def visit(start):
        if start == n:
            out.append(path[:])
            return
        for end in range(start, n):
            if palindrome[start][end]:
                path.append(s[start:end + 1])
                visit(end + 1)
                path.pop()
    visit(0)
    return out
```

- `palindrome=[[False]*n for _ in range(n)]`：建 n×n 的表，行列含义是区间左右端点；注意必须用列表推导建行，直接 `[[False]*n]*n` 会让所有行共享同一个列表。
- `for left in range(n-1,-1,-1)`：left 从最右往左扫。为什么要倒着来？因为递推公式要用 `palindrome[left+1][right-1]`（更靠右的 left），先把右边算好，左边才能查到。
- `s[left]==s[right] and (right-left<2 or palindrome[left+1][right-1])`：两端必须相等；`right-left<2` 表示段长 1 或 2（单字符或恰好一对），这时内部没有字符可看；更长的话就查"去掉两端后的内部段"是否回文。
- `if start==n: out.append(path[:])`：切到字符串末尾说明所有字符都进了合法段，收集 path 拷贝。
- `for end in range(start,n)` 与 `if palindrome[start][end]`：枚举下一段终点，查表 O(1) 判断合法性；不合法直接不进。
- `path.append(s[start:end+1]); visit(end+1); path.pop()`：段是左闭右开切片，下一层从 end+1 开始；三部曲收尾。

### 4.2 `restore_ip_addresses(s)`

（对应程序见下方分段实现与完整代码。）

- `if any(c not in '0123456789' for c in s): raise ValueError(...)`：输入契约——只接受 ASCII 数字，混入字母或空格立刻报错，因为后面的 `int(part)` 依赖这一点。
- `needed=4-len(path); remaining=len(s)-start`：还需要几段、还剩几个字符。
- `if not needed<=remaining<=3*needed: return`：剩余长度剪枝。每个剩余的段至少吃 1 个字符、至多吃 3 个，剩余字符不在这个区间就不可能成功，直接返回。这一行还顺手处理了"字符串超过 12 个字符"的情况（根部 remaining>12 时直接判死），也保证了走到 `needed==0` 时 remaining 必然是 0（0≤remaining≤0）。
- `if needed==0: out.append('.'.join(path))`：四段凑齐且字符恰好用完（由上一行保证），把段用点拼成地址字符串收集。
- `for length in range(1,4)`：段长只能 1、2、3。
- `if start+length>len(s): break`：不够切了；更长的候选更不够切，所以 break。
- `if len(part)>1 and part[0]=='0': break`：前导零闸门。'0' 本身合法，但 '01'、'00' 非法；一旦长度 2 带头零，长度 3 的更长候选必然也带头零，break。
- `if int(part)>255: break`：数值上限。候选随长度增长，'999' 比 '99' 大，超了就 break。
- `path.append(part); visit(start+length); path.pop()`：三部曲。

注意三个闸门用的都是 `break` 而不是 `continue`：因为"切更长"只会让前导零更严重、数值更大，本候选不行时更长的候选全都不行，这是字符串前缀性质送我们的强化剪枝。

## 5. 分段实现：按顺序运行

**本章接口：** `partition_palindromes(s)`、`restore_ip_addresses(s)`

### partition_palindromes

In [1]:
def partition_palindromes(s):
    n = len(s)
    palindrome = [[False] * n for _ in range(n)]
    for left in range(n - 1, -1, -1):
        for right in range(left, n):
            palindrome[left][right] = s[left] == s[right] and (right - left < 2 or palindrome[left + 1][right - 1])
    out = []
    path = []

    def visit(start):
        if start == n:
            out.append(path[:])
            return
        for end in range(start, n):
            if palindrome[start][end]:
                path.append(s[start:end + 1])
                visit(end + 1)
                path.pop()
    visit(0)
    return out

### restore_ip_addresses

In [2]:
def restore_ip_addresses(s):
    if any((c not in '0123456789' for c in s)):
        raise ValueError('ASCII decimal digits required')
    out = []
    path = []

    def visit(start):
        needed = 4 - len(path)
        remaining = len(s) - start
        if not needed <= remaining <= 3 * needed:
            return
        if needed == 0:
            out.append('.'.join(path))
            return
        for length in range(1, 4):
            if start + length > len(s):
                break
            part = s[start:start + length]
            if len(part) > 1 and part[0] == '0':
                break
            if int(part) > 255:
                break
            path.append(part)
            visit(start + length)
            path.pop()
    visit(0)
    return out

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['切分','段长','全为回文'],[(p,list(map(len,p)),all(x==x[::-1] for x in p)) for p in partition_palindromes('aab')])
show_table(['输入','合法IP'],[('25525511135',restore_ip_addresses('25525511135')),('0000',restore_ip_addresses('0000'))])

切分,段长,全为回文
"['a', 'a', 'b']","[1, 1, 1]",True
"['aa', 'b']","[2, 1]",True


输入,合法IP
25525511135,"['255.255.11.135', '255.255.111.35']"
0000,['0.0.0.0']


## 7. 正确性、适用条件与复杂度

任意切分有唯一递增切点序列，枚举所有合法下一段即可完备。IP长度剪枝只删除不可能把剩余字符放入剩余1..3位段的状态。

**代价：** 回文预处理O(n²)，输出最坏指数；IP至多3⁴条深度4路径，输入长度超过12时可立即排除。

### 展开理解

**回文切分为什么不会漏、不会多？** 任何一种切法都唯一对应一串从小到大的切点位置（比如 'aab' 的 ['a','a','b'] 对应切点 1、2）。我们的递归在每个状态下把"下一段的所有可能终点"枚举了个遍，所以每种切点序列都会被走一次——这保证了不漏；同时只有"当前段是回文"的候选才被允许进入，所以收集到的每种切法必然每段都是回文——这保证了不多。

**IP 剪枝为什么安全？** 剪掉的分支是"剩余字符装不进剩余段"或"剩余字符填不满剩余段"的状态。这种状态无论后面怎么切都不可能变成合法 IP（段长上限 3、段数固定 4 是硬性规则），所以砍掉它们不会误伤任何合法答案。三道闸门同理：前导零和 255 上限是题目规则，挡掉的本来就不是答案。

**复杂度（结合数字说）：** 回文表预计算是 O(n²)——n=16 的字符串也就 256 个格子，眨眼填完。回溯部分最坏是指数级的（比如 'aaaa...a' 每个位置都能切，切法随 n 爆炸），这由答案数量决定，n 大答案多时谁也快不了。IP 这边最多四层、每层至多 3 个分支，总共至多 3⁴=81 条路径，每次还带剪枝，所以哪怕字符串很长（直接被长度剪枝挡在根部）也是瞬间出结果。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 逐条解释：

- `assert sorted(partition_palindromes('aab'))==[['a','a','b'],['aa','b']]`：正常值。这是本题最经典的例子，两条答案与我们的手算演示一致，同时验证了递归顺序无关紧要（用 sorted 比对）。
- `assert set(restore_ip_addresses('25525511135'))=={'255.255.11.135','255.255.111.35'}`：正常值。LeetCode 93 的原例，两条答案分别对应"11.135"和"111.35"两种分段方式，考察 1..3 段长枚举是否齐全。
- `assert restore_ip_addresses('0000')==['0.0.0.0']`：特殊值。专门打前导零规则：四个 0 只能各成一段；如果实现漏了前导零检查，就会多出 '00.0.0.0' 之类的非法答案。
- `for s in ['','a','abba','aabaa']: for parts in partition_palindromes(s): assert ''.join(parts)==s and all(...)`：自我一致性检查。空串、单字符、偶回文、奇回文都过一遍；每条答案把段拼回去必须等于原串（没有丢字符、没有重叠），且每段反转后等于自己（真是回文）。
- `for s in ['101023','1111','00000','1234567']: ...` 的 combinations 循环：**暴力对照测试**。它用 `combinations(range(1,len(s)),3)` 枚举所有可能的三个切点位置，手工拼出四段并按规则过滤，得到参考答案集，再和我们的实现比对。'101023' 是 LeetCode 原题有多解的样例，'00000'（5 个字符，超短但前导零陷阱密集）和 '1234567'（7 个字符）补充覆盖。

In [4]:
assert sorted(partition_palindromes('aab')) == [['a', 'a', 'b'], ['aa', 'b']]

assert set(restore_ip_addresses('25525511135')) == {'255.255.11.135', '255.255.111.35'}

assert restore_ip_addresses('0000') == ['0.0.0.0']

for s in ['', 'a', 'abba', 'aabaa']:
    for parts in partition_palindromes(s):
        assert ''.join(parts) == s and all((x == x[::-1] for x in parts))

from itertools import combinations

for s in ['101023', '1111', '00000', '1234567']:
    ref = []
    for cuts in combinations(range(1, len(s)), 3):
        bounds = (0, *cuts, len(s))
        parts = [s[bounds[i]:bounds[i + 1]] for i in range(4)]
        if all((len(x) <= 3 and (len(x) == 1 or x[0] != '0') and (int(x) <= 255) for x in parts)):
            ref.append('.'.join(parts))
    assert sorted(restore_ip_addresses(s)) == sorted(ref)

print('N066: 所有本章断言通过')

N066: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 预计算回文表作为进阶对照。

**练习 2：** 说明IP片段不能带多余前导零。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（预计算回文表作为进阶对照）**：思路是先写一个"不预计算、每候选现场 `part==part[::-1]` 判断"的朴素版本，再和本章表驱动版本对比。提示：关注两个点——同样输入下两个版本答案是否完全一致（正确性对照），以及当字符串是 'aaaa...a'（每个区间都是回文）时现场判断要重复比较多少次字符（效率差距）。手算示例可以用 'aba'：朴素版判断区间 [0,2] 时比较 2 次字符，表驱动版本查询 1 次。
- **练习 2（说明 IP 片段不能带多余前导零）**：思路是解释规则动机和实现位置。提示：动机层面，'01' 和 '1' 数值都是 1，允许前导零会让同一个地址出现两种写法，甚至 '255.01.1.1' 这种有歧义的串；实现层面，指出代码里 `if len(part)>1 and part[0]=='0': break` 这一行就是闸门，并用手算演示 '0000' 说明为什么是 break——长度 2 的 '00' 带头零时，长度 3 的 '000' 必然也带。你可以临时删掉这行跑 '0000'，看看输出多出哪些非法答案。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def partition_palindromes(s):
    n = len(s)
    palindrome = [[False] * n for _ in range(n)]
    for left in range(n - 1, -1, -1):
        for right in range(left, n):
            palindrome[left][right] = s[left] == s[right] and (right - left < 2 or palindrome[left + 1][right - 1])
    out = []
    path = []

    def visit(start):
        if start == n:
            out.append(path[:])
            return
        for end in range(start, n):
            if palindrome[start][end]:
                path.append(s[start:end + 1])
                visit(end + 1)
                path.pop()
    visit(0)
    return out

def restore_ip_addresses(s):
    if any((c not in '0123456789' for c in s)):
        raise ValueError('ASCII decimal digits required')
    out = []
    path = []

    def visit(start):
        needed = 4 - len(path)
        remaining = len(s) - start
        if not needed <= remaining <= 3 * needed:
            return
        if needed == 0:
            out.append('.'.join(path))
            return
        for length in range(1, 4):
            if start + length > len(s):
                break
            part = s[start:start + length]
            if len(part) > 1 and part[0] == '0':
                break
            if int(part) > 255:
                break
            path.append(part)
            visit(start + length)
            path.pop()
    visit(0)
    return out

# 示例与回归测试
assert sorted(partition_palindromes('aab')) == [['a', 'a', 'b'], ['aa', 'b']]

assert set(restore_ip_addresses('25525511135')) == {'255.255.11.135', '255.255.111.35'}

assert restore_ip_addresses('0000') == ['0.0.0.0']

for s in ['', 'a', 'abba', 'aabaa']:
    for parts in partition_palindromes(s):
        assert ''.join(parts) == s and all((x == x[::-1] for x in parts))

from itertools import combinations

for s in ['101023', '1111', '00000', '1234567']:
    ref = []
    for cuts in combinations(range(1, len(s)), 3):
        bounds = (0, *cuts, len(s))
        parts = [s[bounds[i]:bounds[i + 1]] for i in range(4)]
        if all((len(x) <= 3 and (len(x) == 1 or x[0] != '0') and (int(x) <= 255) for x in parts)):
            ref.append('.'.join(parts))
    assert sorted(restore_ip_addresses(s)) == sorted(ref)

print('N066: 所有本章断言通过')

N066: 所有本章断言通过


## 11. 代表题与迁移

- **131. Palindrome Partitioning**：对应 `partition_palindromes`，练的是"切分即选择"的状态设计，以及"先 O(n²) 预计算回文表再回溯查表"的两段式优化。
- **93. Restore IP Addresses**：对应 `restore_ip_addresses`，练的是"多约束闸门 + 剩余长度剪枝"——约束多的切分题要把每条规则翻译成一句 `break`/`return`。

这两题告诉你：切分类回溯的骨架完全一样（状态=已切到哪，选择=下一段终点），题目难度全在"下一段合法"的判断上。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。